> **한눈에 보기**
> - 클라우드 벡터 DB **Pinecone**을 LangChain 없이 SDK로 직접 사용
> - 인덱스 생성(1536차원, cosine) → OpenAI 임베딩 upsert → 질문 벡터로 query
> - 정리 문서: [크로마벡터스토어.md](크로마벡터스토어.md)

### 1. 준비

In [1]:
# .env의 PINECONE_API_KEY, OPENAI_API_KEY 불러오기
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import os
from pinecone import Pinecone, ServerlessSpec

### 2. 인덱스 만들기/연결

In [3]:
# 인덱스 = Pinecone 안의 벡터 저장 공간. dimension은 임베딩 모델 차원과 같아야 한다
# ServerlessSpec: 서버를 직접 두지 않는 서버리스 방식 (AWS us-east-1)
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

index_name = "quickstart-index"

# 인덱스가 없는 경우 새로 생성 (OpenAI text-embedding-3-small 기준 1536차원)
if index_name not in [index.name for index in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

# 인덱스 연결 
index = pc.Index(index_name) 

### 3. 저장(upsert)하고 검색(query)하기

In [5]:
from openai import OpenAI
import time

openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# 1. 테스트할 문서
documents = [
    {"id": "doc1", "text": "Pinecone은 벡터 데이터베이스입니다."},
    {"id": "doc2", "text": "Pinecone은 임베딩 벡터를 저장하고 유사한 정보를 검색하는 서비스입니다."},
]

# 2. 문장을 벡터로 변환해서 Pinecone에 저장
response = openai_client.embeddings.create(
    model="text-embedding-3-small",
    input=[document["text"] for document in documents],
)

vectors = [
    {
        "id": document["id"],
        "values": embedding.embedding,
        "metadata": {"text": document["text"]},
    }
    for document, embedding in zip(documents, response.data)
]
# 동작 방식 (Upsert = Update + Insert) : id중복 요청시 오류없이 덮어씀.
index.upsert(vectors=vectors)
time.sleep(2)

# 3. 검색용 질문도 같은 방식으로 벡터 변환
question = "임베딩 데이터를 저장하고 유사한 정보를 찾는 서비스는 무엇인가요?"
question_vector = openai_client.embeddings.create(
    model="text-embedding-3-small",
    input=question,
).data[0].embedding

# 4. 질문과 의미가 가까운 문서 검색
result = index.query(
    vector=question_vector,
    top_k=2,
    include_metadata=True,
)

print(f"질문: {question}\n") 
for match in result.matches:
    print(f"{match.id}: {match.metadata['text']} (유사도: {match.score:.4f})") 

질문: 임베딩 데이터를 저장하고 유사한 정보를 찾는 서비스는 무엇인가요?

doc2: Pinecone은 임베딩 벡터를 저장하고 유사한 정보를 검색하는 서비스입니다. (유사도: 0.6642)
doc1: Pinecone은 벡터 데이터베이스입니다. (유사도: 0.4797)


**📝 정리** — 질문과 표현이 더 많이 겹치는 doc2(0.6642)가 doc1(0.4797)보다 위에 왔다. `time.sleep(2)`는 upsert한 벡터가 검색에 반영될 때까지 잠깐 기다리는 것이다.

## 🔁 복습 질문

<details>
<summary>1. Pinecone 인덱스의 <code>dimension</code>을 1536으로 한 이유는?</summary>

저장할 벡터를 만드는 `text-embedding-3-small`의 차원이 1536이기 때문이다. 다르면 저장할 수 없다.
</details>

<details>
<summary>2. <code>upsert</code>는 무엇인가?</summary>

update + insert. 같은 id가 있으면 덮어쓰고 없으면 새로 넣는다.
</details>

<details>
<summary>3. Chroma·FAISS와 Pinecone의 차이는?</summary>

Chroma·FAISS는 내 컴퓨터에서 돌고, Pinecone은 클라우드 서비스라 API 키로 접속하며 데이터가 계정에 계속 남는다.
</details>
